# DF-A Ders Kitabı: Araç Piyasası Verisiyle Tanışma

Bu notebook, veri bilimi geçmişin olmasa bile baştan sona takip edebileceğin
şekilde yazıldı. Her adımda önce **ne yaptığımızı günlük dille** anlatıyoruz,
sonra **kodu** çalıştırıyoruz, sonra da **sonucun ne anlama geldiğini**
yorumluyoruz. Yeni bir terimle karşılaştığında 📖 işaretli kutularda
tanımını bulacaksın.

**Notebook'un ele aldığı veri seti:** `df_a_kapsama_testli_v2.csv` (kısaca
"DF-A" diyeceğiz).


---
## Bölüm 1 — Bu Veri Seti Ne, Neden Var?

Projenin genel amacı şu: **Türkiye'de araç piyasasının fiyat yönünü
(artacak mı, azalacak mı, aynı mı kalacak) önceden tahmin edebilecek bir
sistem kurmak.** Bunu yapabilmek için önce elimizde piyasayı etkileyebilecek
göstergelerden oluşan temiz bir tablo olması gerekiyor — işte DF-A, bu
tablolardan biri.

**DF-A'nın özel bir kuralı var:** İçindeki her sütun, DF-A'nın kapsadığı
TÜM zaman aralığında (neredeyse) kesintisiz doludur. Bunu şöyle sağladık:
elimizdeki "noter devri" verisinin (bir ayda kaç aracın el değiştirdiğinin
kaydı) otomobile özgü versiyonu ne zaman başlıyorsa, DF-A da tam o tarihte
başlıyor. Bu tarihten daha geç başlayan hiçbir kaynak (örneğin BETAM adlı
bir şirketin ikinci el ilan fiyatı verisi, ya da ENAG adlı bir grubun
enflasyon verisi) DF-A'ya alınmadı — çünkü onlar daha sonra (2024'ten
itibaren) başlıyor ve DF-A'yı "delik deşik" ederlerdi.

*(Bu yüzden ayrıca bir de DF-B var: DF-B, daha KISA bir zaman aralığını
[2024'ten bugüne] kapsıyor ama karşılığında BETAM'ın fiyat verisini ve ENAG'ın
enflasyon verisini de İÇERİYOR. DF-A "uzun ama dar", DF-B "kısa ama geniş"
diye düşünebilirsin.)*

Şimdi veriyi yükleyip ilk bakışı atalım.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sayfada daha fazla sütun/genişlik gösterebilmek icin ayar
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# DF-A'yi diskten oku
df_a = pd.read_csv("../data/processed/dataframes/df_a_kapsama_testli_v2.csv")

# "referans_ayi" sutunu "2018-01" gibi bir metin - bunu gercek bir tarih
# olarak da tutalim ki grafiklerde duzgun gorunsun
df_a["tarih"] = pd.to_datetime(df_a["referans_ayi"])

# Kac satir, kac sutun oldugunu goster
print("Boyut (satir, sutun):", df_a.shape)
print("Kapsadigi tarih araligi:", df_a["referans_ayi"].min(), "->", df_a["referans_ayi"].max())


**Bu sonuç bize ne söylüyor?**

DF-A, 102 satırdan (yani 102 AY'dan — 8,5 yıla yakın bir süre) ve 16
sütundan oluşuyor. Kapsadığı aralık **2018-01'den 2026-06'ya** kadar.
Dikkat: bu, projenin bazı diğer tablolarının kapsadığı 2015'e kadar
GERİYE GİTMİYOR — çünkü DF-A'nın "çapa" sütunu (otomobile özgü noter
devri) ancak 2018'den itibaren tam veriyle mevcut. Bu kısıtlama bilinçli
bir tercih, veri kaybı değil (bkz. Bölüm 4).


In [ ]:
# Ilk 5 satira bakalim - tabloya alisalim
df_a.head()


**Bu sonuç bize ne söylüyor?**

Her satır bir AYI temsil ediyor (`referans_ayi` sütunu, ör. "2018-01" =
2018 yılının Ocak ayı). Her sütun ise o ayda ölçülmüş farklı bir gösterge
— döviz kuru, enflasyon, faiz oranları, üretim/satış rakamları, tüketici
anketleri ve en sonunda projenin asıl ilgilendiği "noter devri" (el
değiştiren araç sayısı). Bölüm 2'de bu sütunların HER BİRİNİ tek tek
tanıyacağız.


---
## Bölüm 2 — Sütun Sütun Tanışma

Şimdi DF-A'daki her sütunu sırayla ele alacağız. Her biri için: ne ölçtüğü,
hangi kurumdan geldiği, basit bir istatistik özeti ve zaman içindeki
seyrini gösteren bir grafik.

> 📖 **Terim — "İstatistik özeti" (`describe()`):** Bir sütundaki
> sayıların genel görünümünü tek bakışta veren bir özet. En önemli 3
> tanesi: **ortalama** (tüm değerlerin toplamının sayıya bölünmüşü — "tipik"
> değer), **min/max** (en küçük ve en büyük değer — uç noktalar), **std**
> (standart sapma — değerlerin ortalamadan ne kadar "sağa sola saptığı";
> büyükse seri oynak/değişken demektir, küçükse seri istikrarlı demektir).


### `usdtry_aysonu`

**Ne ölçüyor?** Ayın son iş gününde 1 ABD dolarının kaç Türk Lirası ettiği — yani "ay sonu dolar kuru".

**Kaynak:** TCMB (Türkiye Cumhuriyet Merkez Bankası)


In [ ]:
print(df_a["usdtry_aysonu"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["usdtry_aysonu"], marker="o", markersize=2)
plt.title("usdtry_aysonu - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("usdtry_aysonu")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Bu, 2018'den 2026'ya kurun **yaklaşık 12 katına** çıktığını gösteriyor (3,78 TL'den 46,60 TL'ye). Bu, dönem boyunca Türk Lirası'nın dolar karşısında ciddi değer kaybettiğinin doğrudan kanıtı — araç fiyatları gibi ithal bileşeni yüksek ürünlerde bunun etkisini beklemek mantıklı.


### `usdtry_ortalama`

**Ne ölçüyor?** Ayın TÜM iş günlerindeki dolar kurunun ortalaması (yalnızca ay-sonu değil, ayın tamamının genel seviyesi).

**Kaynak:** TCMB


In [ ]:
print(df_a["usdtry_ortalama"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["usdtry_ortalama"], marker="o", markersize=2)
plt.title("usdtry_ortalama - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("usdtry_ortalama")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Neredeyse `usdtry_aysonu` ile birebir aynı şekilde hareket ediyor (bu normal — ikisi de aynı kur serisinin farklı özetleri, aralarında büyük bir fark beklenmez).


### `tufe_endeks`

**Ne ölçüyor?** Tüketici Fiyat Endeksi — market, kira, ulaşım gibi birçok kalemin fiyatının zamanla ne kadar arttığını gösteren resmi, tek bir sayı haline getirilmiş enflasyon ölçüsü.

**Kaynak:** TÜİK (TCMB EVDS üzerinden)


In [ ]:
print(df_a["tufe_endeks"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["tufe_endeks"], marker="o", markersize=2)
plt.title("tufe_endeks - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("tufe_endeks")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** TÜFE de dönem boyunca yaklaşık **12,5 katına** çıkmış — kur ile neredeyse aynı oranda büyümüş olması tesadüf değil: TL değer kaybı ile enflasyon Türkiye'de birbirini besleyen iki olgu.


### `tufe_aylik_degisim`

**Ne ölçüyor?** TÜFE'nin BİR ÖNCEKİ AYA göre yüzde değişimi — yani "bu ay enflasyon ne kadar oldu".

**Kaynak:** Türetilmiş (TÜFE'den hesaplanmış)


In [ ]:
print(df_a["tufe_aylik_degisim"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["tufe_aylik_degisim"], marker="o", markersize=2)
plt.title("tufe_aylik_degisim - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("tufe_aylik_degisim")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Ortalama ayda **%2,5** civarında bir fiyat artışı var — bu küçük görünse de yıl boyunca üst üste binince devasa bir enflasyona denk gelir. En yüksek nokta %13,6 (2021-12, TL'de büyük bir kur şokunun yaşandığı ay) — en düşük nokta ise -%1,4, yani fiyatların GERÇEKTEN gerilediği nadir bir ay.


### `tufe_yillik_degisim`

**Ne ölçüyor?** TÜFE'nin 12 AY ÖNCEKİ aynı aya göre yüzde değişimi — haberlerde en çok duyduğumuz "yıllık enflasyon" rakamı.

**Kaynak:** Türetilmiş (TÜFE'den hesaplanmış)


In [ ]:
print(df_a["tufe_yillik_degisim"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["tufe_yillik_degisim"], marker="o", markersize=2)
plt.title("tufe_yillik_degisim - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("tufe_yillik_degisim")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Dönem boyunca yıllık enflasyon **%8,6 ile %85,5 arasında** salınmış — Türkiye ekonomisinin bu 8,5 yılda çok farklı enflasyon rejimlerinden (düşükten çok yükseğe) geçtiğini gösteriyor.


### `tasit_kredisi_faiz`

**Ne ölçüyor?** Bankaların araç kredisi için müşteriye uyguladığı ortalama yıllık faiz oranı.

**Kaynak:** TCMB


In [ ]:
print(df_a["tasit_kredisi_faiz"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["tasit_kredisi_faiz"], marker="o", markersize=2)
plt.title("tasit_kredisi_faiz - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("tasit_kredisi_faiz")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Araç kredisiyle araba almanın maliyeti de zamanla ciddi yükselmiş (%14'ten %40'a) — bu, kredi ile araç almayı isteyenler için caydırıcı bir etki yaratmış olabilir.


### `politika_faizi`

**Ne ölçüyor?** Merkez Bankası'nın belirlediği ana ("politika") faiz oranı — diğer birçok faizin temelini oluşturur.

**Kaynak:** TCMB


In [ ]:
print(df_a["politika_faizi"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["politika_faizi"], marker="o", markersize=2)
plt.title("politika_faizi - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("politika_faizi")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Bu da benzer bir yükseliş göstermiş, hatta bir noktada %51'e kadar çıkmış (muhtemelen enflasyonla mücadele için sıkılaştırma dönemi) — taşıt kredisi faiziyle birlikte hareket etmesi beklenir, çünkü bankalar kredi faizlerini politika faizine göre belirler.


### `odmd_otomobil_adet`

**Ne ölçüyor?** O ay Türkiye'de kaç adet SIFIR KM (yeni) binek otomobil SATILDIĞI.

**Kaynak:** ODMD (Otomotiv Distribütörleri Derneği)


In [ ]:
print(df_a["odmd_otomobil_adet"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["odmd_otomobil_adet"], marker="o", markersize=2)
plt.title("odmd_otomobil_adet - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("odmd_otomobil_adet")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Ay-ay çok oynak bir seri (9.661'den 146.319'a kadar geniş bir aralık) — hem mevsimsel (yıl sonu kampanyaları gibi) hem de ekonomik koşullara (COVID, kur şokları) bağlı iniş-çıkışlar var.


### `osd_binek_adet`

**Ne ölçüyor?** O ay Türkiye'de kaç adet binek otomobil ÜRETİLDİĞİ (satış değil, fabrikadan çıkan araç sayısı).

**Kaynak:** OSD (Otomotiv Sanayii Derneği)


In [ ]:
print(df_a["osd_binek_adet"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["osd_binek_adet"], marker="o", markersize=2)
plt.title("osd_binek_adet - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("osd_binek_adet")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** İlginç bir bulgu: dönemin BAŞINDAKİ üretim (85.368), dönemin SONUNDAKİ üretimden (67.113) daha YÜKSEK — yani üretim genel olarak bir miktar GERİLEMİŞ. Bunun kesin nedenini bu veriden çıkaramayız (ihracat önceliklendirmesi, tedarik zinciri sorunları gibi farklı açıklamalar olabilir) ama dikkat çekici bir gözlem.


### `tuketici_guven_endeksi`

**Ne ölçüyor?** İnsanların genel ekonomiye ne kadar güvendiğini ölçen bir anket endeksi (0-200 arası, 100 nötr eşiktir).

**Kaynak:** TÜİK/TCMB Tüketici Eğilim Anketi


In [ ]:
print(df_a["tuketici_guven_endeksi"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["tuketici_guven_endeksi"], marker="o", markersize=2)
plt.title("tuketici_guven_endeksi - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("tuketici_guven_endeksi")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Ortalama 80,5 — yani bu endeks NÖTR EŞİK olan 100'ün SÜREKLİ altında kalmış. Bu, anket katılımcılarının dönem boyunca genel olarak ekonomiye karamsar baktığı anlamına geliyor.


### `otomobil_satinalma_ihtimali_endeksi`

**Ne ölçüyor?** Ankete katılanlara sorulan "önümüzdeki 12 ayda otomobil satın alma ihtimaliniz nedir" sorusunun endeksi.

**Kaynak:** Aynı anket (TÜİK/TCMB), araca özel soru


In [ ]:
print(df_a["otomobil_satinalma_ihtimali_endeksi"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["otomobil_satinalma_ihtimali_endeksi"], marker="o", markersize=2)
plt.title("otomobil_satinalma_ihtimali_endeksi - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("otomobil_satinalma_ihtimali_endeksi")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Bu endeks dönem boyunca neredeyse **2 katına** çıkmış (13,3'ten 26,9'a) — yani insanların "araba alma niyeti" zamanla artmış GÖRÜNÜYOR. Bunun fiyatların da aynı dönemde ciddi arttığı gerçeğiyle birlikte nasıl yorumlanacağı (niyet mi arttı, yoksa insanlar zaten aldıklarını erken almaya mı yöneldi) ekip lideriyle tartışılmaya değer bir bulgu.


### `noter_devir_toplam_adet`

**Ne ölçüyor?** O ay noterde el değiştiren (yani İKİNCİ EL satılan) TÜM taşıtların (otomobil + diğer araç tipleri) sayısı.

**Kaynak:** TÜİK "Motorlu Kara Taşıtları" bültenleri


In [ ]:
print(df_a["noter_devir_toplam_adet"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["noter_devir_toplam_adet"], marker="o", markersize=2)
plt.title("noter_devir_toplam_adet - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("noter_devir_toplam_adet")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Bu sütunu ve otomobile özgü versiyonunu Bölüm 4'te ayrıca ve derinlemesine ele alacağız — çünkü projenin asıl ilgilendiği gösterge bu.


### `noter_devir_otomobil_adet`

**Ne ölçüyor?** Yukarıdakiyle aynı ama YALNIZCA binek otomobillerin el değiştirme sayısı (diğer araç tipleri hariç).

**Kaynak:** Aynı TÜİK bültenleri


In [ ]:
print(df_a["noter_devir_otomobil_adet"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_a["tarih"], df_a["noter_devir_otomobil_adet"], marker="o", markersize=2)
plt.title("noter_devir_otomobil_adet - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("noter_devir_otomobil_adet")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Bölüm 4'te derinlemesine incelenecek.


### `tufe_yayim_tarihi` ve `alim_gucu_ceyrek` — iki "metadata" sütunu

Bu ikisi diğerlerinden farklı: bunlar bir ÖLÇÜM değil, bir **etiket/not**
sütunu.

- **`tufe_yayim_tarihi`**: TÜİK'in o ayın TÜFE rakamını hangi (yaklaşık)
  tarihte açıkladığını gösteren bir metin (genelde bir sonraki ayın 3'ü
  civarı).
- **`alim_gucu_ceyrek`**: Bu sütun aslında, tablodan daha önce çıkarılmış
  bir "alım gücü" ölçümünün hangi ÇEYREĞE (3 aylık döneme, ör. "2018-Q1")
  ait olduğunu gösteriyordu. O ölçüm sütunu artık DF-A'da yok (ayrı bir
  temizlik kararıyla çıkarıldı), bu yüzden bu etiket sütunu şu an
  **"yetim"** durumda — kendi başına bir anlam ifade etmiyor, yalnızca
  geçmişten kalan bir iz. Bunu bilinçli olarak silmedik, ama analiz
  yaparken bu sütunu görmezden gelmen doğru olur.


In [ ]:
print(df_a[["tufe_yayim_tarihi", "alim_gucu_ceyrek"]].head())
print()
print("alim_gucu_ceyrek bos hucre sayisi:", df_a["alim_gucu_ceyrek"].isna().sum())


**Bu sonuç bize ne söylüyor?** `alim_gucu_ceyrek`'te 3 tane boş hücre
var — bunlar 2026'nın henüz tamamlanmamış son çeyreğine (Nisan-Haziran)
denk geliyor, TÜİK bu döneme dair veriyi henüz yayımlamadı. Bu bir hata
değil, sadece "henüz zamanı gelmedi" durumu.


---
## Bölüm 3 — 📖 Temel Kavram: Zaman Serisi Nedir?

Diyelim ki her sabah kendini tartıp kilonu bir deftere yazıyorsun. Bir ay
sonra elinde 30 tane sıralı sayı olur — işte buna **zaman serisi** denir:
"aynı şeyin, düzenli aralıklarla, zaman içinde tekrar tekrar ölçülmesiyle
oluşan sıralı bir veri dizisi."

> 📖 **Terim — Zaman Serisi:** Aynı göstergenin (kilo, döviz kuru, satış
> adedi...) belirli aralıklarla (her gün, her ay, her yıl) tekrar tekrar
> ölçülmesiyle oluşan, SIRASI ÖNEMLİ olan bir veri dizisi. Sırası önemlidir
> çünkü bir ayın değeri genelde bir önceki aya bağlıdır — kilon dün 70 kg
> idiyse bugün aniden 120 kg olması beklenmez, ama iki yıl arayla büyük
> fark olabilir.

**Bizim verimiz neden bir zaman serisi?** DF-A'daki her satır bir AYI
temsil ediyor ve satırlar TARİH SIRASINA göre dizili (2018-01, 2018-02,
2018-03, ...). Bu ayların sırasını karıştırırsak (mesela 2023-06'yı
2018-01'in yanına koysak) veri anlamsızlaşır — çünkü "bu ay geçen aya göre
ne kadar değişti" gibi sorular ancak doğru sırayla cevaplanabilir.

> 📖 **Terim — Trend:** Bir zaman serisinin UZUN VADEDE genel olarak
> yukarı mı gittiği, aşağı mı gittiği, yoksa aynı seviyede mi kaldığı.
> Örnek: Boyun her yıl biraz uzuyorsa (çocukken), bu YUKARI bir trend'dir.
> Ay-ay küçük iniş çıkışlar olsa bile, genel yön yukarıysa yine trend
> yukarıdır.

> 📖 **Terim — Mevsimsellik:** Bir serinin YIL İÇİNDE belirli aylarda
> DÜZENLİ OLARAK tekrar eden bir örüntü göstermesi. Örnek: Dondurma
> satışları her yıl yaz aylarında artar, kışın azalır — bu her yıl
> TEKRARLANAN bir örüntü olduğu için mevsimsellik denir (rastgele değil,
> takvime bağlı).

Bizim verimizde de hem trend (örneğin dolar kuru sürekli yukarı gitmiş)
hem de mevsimsellik (örneğin araç satışları yıl sonunda genelde artar)
olabilir — Bölüm 4'te noter devri özelinde buna bakacağız.


---
## Bölüm 4 — Hedefimiz: Noter Devir Adedi

Bu projenin can alıcı noktasındayız. `noter_devir_toplam_adet` ve
`noter_devir_otomobil_adet`, o ay Türkiye'de kaç aracın (sırasıyla: tüm
araç tipleri / yalnızca otomobil) noter aracılığıyla EL DEĞİŞTİRDİĞİNİ
(yani ikinci el olarak satıldığını) gösteriyor.

**Bu neden bir "hedef" olabilir?** Projenin nihai amacı "araç piyasasında
fiyat/hareketlilik yönünü önceden görebilmek" ise, noter devri gibi bir
işlem-hacmi göstergesi, piyasanın ne kadar "canlı" ya da "durgun" olduğunu
doğrudan yansıtır — bu ay çok araç el değiştiriyorsa piyasa hareketli,
az değiştiriyorsa durgun demektir. Bunu ileride tahmin etmeye çalışmak,
projenin sorduğu ana soruya (yön ne olacak?) bir cevap olabilir.


In [ ]:
# Yillik ortalama, en dusuk ve en yuksek degerleri hesapla - genel egilimi gormek icin
df_a["yil"] = df_a["tarih"].dt.year
yillik_ozet = df_a.groupby("yil")[["noter_devir_toplam_adet", "noter_devir_otomobil_adet"]].agg(["mean", "min", "max"])
print(yillik_ozet)


**Bu sonuç bize ne söylüyor?** Yıllık ortalamalara bakınca, 2018'de
yaklaşık 644 bin olan toplam devir, 2025'e gelindiğinde yaklaşık 934 bine
çıkmış — yani genel olarak **hafif bir yukarı eğilim (trend)** var. AMA
dikkat: her yılın kendi içindeki min-max farkı da ÇOK BÜYÜK (örneğin 2020
yılında en düşük ay 348 bin, en yüksek ay 1 milyon 97 bin) — yani bu seri
kur/enflasyon gibi "sürekli ve düzgün" bir şekilde artmıyor, çok inişli
çıkışlı (oynak) bir seyir izliyor.


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df_a["tarih"], df_a["noter_devir_toplam_adet"], label="Toplam (tum arac tipleri)", linewidth=1.5)
plt.plot(df_a["tarih"], df_a["noter_devir_otomobil_adet"], label="Yalnizca otomobil", linewidth=1.5)
plt.title("Noter Devir Adedi - Zaman Icinde Seyri (2018-2026)")
plt.xlabel("Tarih")
plt.ylabel("Adet")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Grafikte net bir "düz çizgi halinde
sürekli artış" GÖRMÜYORSUN — bunun yerine keskin iniş-çıkışlarla dolu,
dalgalı bir çizgi var. En dikkat çekici düşüş 2020'nin başında (COVID-19
kapanmaları döneminde piyasa neredeyse durmuş) hemen ardından sert bir
toparlanma var.

> 📖 **"Hep Yukarı" Tuzağı Nedir?** Bazı seriler (mesela bu projede dolar
> kuru veya TÜFE gibi) neredeyse HİÇ DÜŞMEDEN, sürekli yukarı gider. Böyle
> bir seriyi tahmin etmeye çalışan tembel bir yöntem şunu yapabilir: "Ben
> her ay için sadece 'artacak' derim" — ve bu, kur/TÜFE gibi seriler için
> neredeyse hep DOĞRU çıkar, ama aslında hiçbir şey "öğrenmemiş", "tahmin
> etmemiş" olur; sadece serinin doğasından (hep yukarı gitme eğiliminden)
> faydalanmıştır. Böyle bir modelin gerçek bir başarı gibi görünüp
> aslında değersiz olması riskine **"hep yukarı" tuzağı** diyoruz.
>
> **Noter devri için bu tuzak ne kadar geçerli?** Yukarıdaki grafikte
> gördüğün gibi, noter devri kur/TÜFE kadar "düz ve kesintisiz" artmıyor —
> ciddi ay-ay iniş-çıkışları var (COVID düşüşü gibi). Bu yüzden "hep
> artacak" demek burada kur/TÜFE'deki kadar kolay/otomatik doğru çıkmaz —
> ama yine de hafif bir yukarı eğilim olduğu için bu tuzağa TAMAMEN bağışık
> da değil. Herhangi bir model kurulurken bu ayrım (sürekli-artan seri mi,
> yoksa dalgalı-ama-hafif-artan seri mi) mutlaka göz önünde bulundurulmalı.


---
## Bölüm 5 — 📖 Temel Kavram: Korelasyon Nedir?

Yaz aylarında hem dondurma satışları artar hem de deniz kenarındaki
boğulma vakaları artar. Bu ikisi "birlikte" hareket ediyor — biri artarken
diğeri de artıyor. İşte iki şeyin BİRLİKTE aynı yönde (ya da ters yönde)
hareket etme eğilimine **korelasyon** diyoruz.

> 📖 **Terim — Korelasyon Katsayısı (r):** İki serinin ne kadar "birlikte
> hareket ettiğini" -1 ile +1 arasında bir sayıyla ölçen bir değer.
> - **+1'e yakın** → ikisi neredeyse HEP BİRLİKTE artıp azalıyor (güçlü,
>   AYNI YÖNDE ilişki).
> - **-1'e yakın** → biri artarken diğeri neredeyse HEP azalıyor (güçlü,
>   TERS YÖNDE ilişki).
> - **0'a yakın** → aralarında belirgin bir ilişki YOK, birbirinden
>   bağımsız hareket ediyorlar.

**Önemli bir uyarı var:**

> 📖 **Terim — Sahte (Spurious) Korelasyon:** İki şeyin birlikte hareket
> etmesi, HER ZAMAN birinin diğerini ETKİLEDİĞİ anlamına gelmez. Klasik
> bir örnek: Bir şehirde hem "itfaiyeci sayısı" hem de "yangın hasarı
> miktarı" birlikte yüksek çıkabilir — ama itfaiyeciler yangın
> ÇIKARMIYOR, tam tersine söndürüyor! Asıl sebep üçüncü bir şey olabilir
> (örneğin şehrin büyüklüğü — büyük şehirde hem daha çok itfaiyeci HEM de
> daha çok yangın olur). Bizim verimizde de buna benzer bir risk var: kur
> ve TÜFE gibi göstergeler, ikisi de zamanla "genel olarak artan" seriler
> olduğu için, aralarında YÜKSEK bir korelasyon çıkması aslında "biri
> diğerini etkiliyor" değil, "ikisi de aynı genel zaman trendini
> paylaşıyor" anlamına gelebilir. Bölüm 6-8'de bu ayrımı somut olarak
> göreceğiz.


---
## Bölüm 6 — Ham Seviye Korelasyon (ve Neden Tek Başına Yeterli Değil)

Şimdi DF-A'daki TÜM sayısal sütunları birbiriyle karşılaştırıp bir
**korelasyon matrisi** (her sütun çiftinin r değerini gösteren bir tablo)
çıkaralım, sonra bunu bir **ısı haritasıyla (heatmap)** görselleştirelim —
ısı haritasında koyu kırmızı/mavi renkler güçlü ilişkiyi, açık renkler
zayıf ilişkiyi gösterir.

**Burada dikkat:** Bu ilk denemede sütunları OLDUKLARI GİBİ (ham seviye
değerleriyle) kullanacağız — yani kurun kendisini, TÜFE'nin kendisini vb.
Bunun neden sorunlu olabileceğini az sonra göreceğiz.


In [ ]:
numeric_cols = ['usdtry_aysonu', 'usdtry_ortalama', 'tufe_endeks', 'tufe_aylik_degisim', 'tufe_yillik_degisim', 'tasit_kredisi_faiz', 'politika_faizi', 'odmd_otomobil_adet', 'osd_binek_adet', 'tuketici_guven_endeksi', 'otomobil_satinalma_ihtimali_endeksi', 'noter_devir_toplam_adet', 'noter_devir_otomobil_adet']

korelasyon_ham = df_a[numeric_cols].corr()

plt.figure(figsize=(9, 8))
im = plt.imshow(korelasyon_ham.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90, fontsize=8)
plt.yticks(range(len(numeric_cols)), numeric_cols, fontsize=8)
plt.colorbar(im, label="Korelasyon (r)")
plt.title("Ham Seviye Korelasyon Matrisi")
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?**

En yüksek (|r| > 0,8) çıkan bazı çiftler:

| Çift | r |
|---|---|
| usdtry_aysonu ↔ usdtry_ortalama | 0,9996 |
| usdtry_ortalama ↔ tufe_endeks | 0,984 |
| noter_devir_toplam_adet ↔ noter_devir_otomobil_adet | 0,983 |
| usdtry_aysonu ↔ tufe_endeks | 0,983 |
| tasit_kredisi_faiz ↔ politika_faizi | 0,925 |
| tufe_endeks ↔ otomobil_satinalma_ihtimali_endeksi | 0,920 |
| usdtry_aysonu ↔ otomobil_satinalma_ihtimali_endeksi | 0,880 |
| usdtry_ortalama ↔ tasit_kredisi_faiz | 0,874 |

`usdtry_aysonu ↔ tufe_endeks` r=0,983 gibi çok yüksek bir sayı görünce ilk
tepki "kur arttıkça enflasyon da artıyor, aralarında güçlü bir ilişki
var!" demek olabilir. AMA Bölüm 5'teki uyarıyı hatırla: **ikisi de zaten
zamanla sürekli büyüyen (trend taşıyan) seriler** — ikisinin de "seviyesi"
her ay biraz daha yüksek olduğu için otomatik olarak yüksek korelasyon
çıkar, bu illa ki "biri diğerini etkiliyor" demek değildir (muhtemelen
GERÇEKTEN ilişkililer ama BU HALİYLE ÖLÇTÜĞÜMÜZ korelasyon abartılı
olabilir). Bölüm 7-8'de bunu "adil" bir şekilde nasıl ölçebileceğimizi
göreceğiz.


---
## Bölüm 7 — 📖 Temel Kavram: Aylık Değişim / Log-Değişim

**Neden "ham seviye" yanıltıcı olabilir?** Yukarıda gördüğümüz gibi, iki
seri sadece ikisi de "genel olarak büyüdüğü" için yüksek korelasyon
gösterebilir. Bunu düzeltmenin yolu: seviyenin KENDİSİNE değil, **"bu ay
geçen aya göre ne kadar değişti"**ye bakmak. Böylece ortak trend etkisi
büyük ölçüde elenir, geriye asıl "birlikte hareket etme" sinyali kalır.

> 📖 **Terim — Aylık Yüzde Değişim:** `(bu ayki değer - geçen ayki değer) /
> geçen ayki değer × 100`. Örnek: geçen ay 100 TL olan bir şey bu ay 110
> TL olduysa, aylık değişim %10'dur.

> 📖 **Terim — Log-Değişim:** Yüzde değişime çok benzeyen, ama biraz farklı
> matematiksel bir formülle (`ln(bu ayki değer / geçen ayki değer)`)
> hesaplanan bir değişim ölçüsü. KÜÇÜK değişimlerde ikisi neredeyse AYNI
> sonucu verir (örneğin %2 ile log-değişim 0,0198 hemen hemen aynı
> şeydir). Log-değişimin tercih edilme sebebi, büyük artışlar ile büyük
> düşüşleri MATEMATİKSEL OLARAK daha "adil"/simetrik şekilde ele
> alması — bu projede istatistik hesaplamalarında (özellikle oynaklık
> ölçümlerinde) genellikle log-değişim tercih ediliyor. Teknik detayına
> girmeden, "yüzde değişimin biraz daha hassas bir kuzeni" diye
> düşünebilirsin.

Şimdi, seviye tipi (kur, TÜFE, faiz, üretim/satış, anket, noter devri)
sütunların HER BİRİ için aylık log-değişimi hesaplayacağız. **Not:**
`tufe_aylik_degisim` ve `tufe_yillik_degisim` sütunlarını bu işleme dahil
etmiyoruz çünkü onlar zaten birer "değişim" ölçüsü — bir değişimin bir
daha değişimini almak kafa karıştırır, onları olduğu gibi bırakıyoruz.

**Önemli:** Bu hesapladığımız log-değişim sütunları yalnızca bu notebook
içinde, GEÇİCİ olarak oluşturuluyor — kalıcı `df_a_kapsama_testli_v2.csv`
dosyasını DEĞİŞTİRMİYORUZ.


In [ ]:
# Log-degisim hesaplanacak sutunlar (seviye tipi olanlar)
seviye_sutunlari = ['usdtry_aysonu', 'usdtry_ortalama', 'tufe_endeks', 'tasit_kredisi_faiz', 'politika_faizi', 'odmd_otomobil_adet', 'osd_binek_adet', 'tuketici_guven_endeksi', 'otomobil_satinalma_ihtimali_endeksi', 'noter_devir_toplam_adet', 'noter_devir_otomobil_adet']

# Gecici bir DataFrame - log-degisimleri burada tutacagiz, orijinal df_a'ya dokunmuyoruz
log_degisim = pd.DataFrame()
log_degisim["tarih"] = df_a["tarih"]

for sutun in seviye_sutunlari:
    # ln(bu_ay / gecen_ay) - ilk ay icin "gecen ay" olmadigindan sonuc bos (NaN) cikar, bu normaldir
    log_degisim[sutun] = np.log(df_a[sutun] / df_a[sutun].shift(1))

print("Gecerli (bos olmayan) satir sayisi:", log_degisim.dropna().shape[0], "/ toplam", len(log_degisim))
log_degisim.head()


**Bu sonuç bize ne söylüyor?** İlk satır (2018-01) her zaman boş (NaN)
çıkar — çünkü ondan önceki ayı (2017-12) bilmiyoruz, değişim
hesaplanamıyor. Bunun dışında 100 ay için geçerli log-değişim değerlerimiz
var — bu, korelasyon hesaplamak için hâlâ yeterince büyük bir sayı.


---
## Bölüm 8 — Log-Değişim Korelasyonu (Asıl Anlamlı Analiz)

Şimdi Bölüm 6'daki ısı haritasını, bu kez HAM SEVİYE yerine LOG-DEĞİŞİM
serileriyle tekrarlayalım.


In [ ]:
korelasyon_log = log_degisim.drop(columns=["tarih"]).corr()

plt.figure(figsize=(9, 8))
im = plt.imshow(korelasyon_log.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(korelasyon_log.columns)), korelasyon_log.columns, rotation=90, fontsize=8)
plt.yticks(range(len(korelasyon_log.columns)), korelasyon_log.columns, fontsize=8)
plt.colorbar(im, label="Korelasyon (r)")
plt.title("Log-Degisim Korelasyon Matrisi")
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor? (Bölüm 6 ile karşılaştırma)**

| Çift | Ham seviye r | Log-değişim r | Ne oldu? |
|---|---|---|---|
| usdtry_ortalama ↔ tufe_endeks | 0,984 | **0,44** | Büyük ölçüde **eridi** — bu ilişkinin çoğu ortak trendten kaynaklanıyormuş |
| usdtry_aysonu ↔ tufe_endeks | 0,983 | *(<0,4, listede yok)* | Neredeyse tamamen **eridi** |
| tasit_kredisi_faiz ↔ politika_faizi | 0,925 | **0,58** | Zayıfladı ama HÂLÂ belirgin — muhtemelen gerçek bir ilişki (banka faizleri politika faizine göre belirlenir) |
| noter_devir_toplam_adet ↔ noter_devir_otomobil_adet | 0,983 | **0,996** | Neredeyse hiç değişmedi — bu şaşırtıcı değil, ikisi zaten aynı olayın (toplam vs. yalnızca otomobil) iki yakın versiyonu |
| osd_binek_adet ↔ noter_devir_toplam_adet | *(listede değildi)* | **0,60** (YENİ ortaya çıktı) | Ham seviyede görünmeyen ama log-değişimde ortaya çıkan bir ilişki — üretim ile ikinci el piyasası birlikte hareket ediyor olabilir |

Bu tablo, Bölüm 5'teki uyarıyı somut olarak doğruluyor: kur↔TÜFE gibi
"ikisi de sürekli büyüyor" ilişkileri log-değişimde büyük ölçüde
KAYBOLDU (sahte korelasyon şüphesi güçlendi) — ama taşıt kredisi↔politika
faizi gibi bazı ilişkiler HÂLÂ güçlü kaldı (gerçek bir ekonomik bağ
olduğuna işaret ediyor).


### Noter devrinin diğer TÜM özelliklerle log-değişim korelasyonu

Şimdi özellikle projenin hedefine (noter devri) odaklanalım — diğer TÜM
özelliklerle log-değişim korelasyonunu sıralayalım.


In [ ]:
for hedef in ["noter_devir_toplam_adet", "noter_devir_otomobil_adet"]:
    print(f"=== {hedef} ile digerlerinin log-degisim korelasyonu (buyukten kucuge) ===")
    diger_sutunlar = [c for c in korelasyon_log.columns if c not in ["noter_devir_toplam_adet", "noter_devir_otomobil_adet"]]
    sonuc = korelasyon_log.loc[hedef, diger_sutunlar].sort_values(key=abs, ascending=False)
    print(sonuc)
    print()


**Bu sonuç bize ne söylüyor?**

Noter devrinin (hem toplam hem otomobil versiyonu) diğer özelliklerle en
güçlü log-değişim ilişkisi **`osd_binek_adet`** (yerli üretim, r≈0,59-0,60)
ve **`odmd_otomobil_adet`** (sıfır km satış, r≈0,46) ile. Bunun mantıklı
bir ekonomik hikayesi olabilir: yeni araç piyasası (üretim + satış)
canlandığında, muhtemelen araç sahipleri eski araçlarını değiştirip
yenisini alıyor — bu da ikinci el (noter devri) hareketliliğini
artırıyor.

Buna karşılık, kur/TÜFE/faiz gibi makro göstergelerin noter devriyle
log-değişim ilişkisi ÇOK ZAYIF (|r| < 0,2) — Bölüm 6'daki ham-seviye
tablosunda bu göstergeler arasında yüksek korelasyonlar görmüştük, ama bu
büyük ölçüde ortak trend etkisiymiş; gerçek ay-ay hareketlere bakınca
noter devriyle aralarında güçlü bir bağ YOK.


---
## Bölüm 9 — Stratejik Çıkarımlar

Bu notebook'ta öğrendiklerimizi sade bir dille toparlayalım. **Not:** Bu
bölüm bir KARAR belgesi değil — hangi özelliğin kullanılacağı, model
kurulup kurulmayacağı gibi kararlar proje sahibine ait. Burada yalnızca
bulguları özetliyoruz.

**1) Ham seviye korelasyona güvenme, önce log-değişime bak.**
Kur, TÜFE gibi seriler ikisi de "hep büyüyor" olduğu için aralarında çok
yüksek (ör. r=0,98) korelasyon çıkar — ama bu büyük ölçüde SAHTE
(ortak trend kaynaklı). Log-değişime geçince bu ilişkilerin çoğu büyük
ölçüde zayıflıyor veya kayboluyor.

**2) Noter devriyle en tutarlı (log-değişimde de ayakta kalan) ilişkiler:
`osd_binek_adet` (yerli üretim) ve `odmd_otomobil_adet` (sıfır km
satış).** Bu ikisi, hem ham seviyede hem log-değişimde noter devriyle
birlikte hareket ediyor — bu, projenin ileride feature seçimi yaparken
öncelik verebileceği iki aday.

**3) Kur/TÜFE/faiz gibi makro göstergelerin noter devriyle ay-ay ilişkisi
zayıf.** Bunlar ham seviyede her şeyle yüksek korelasyon gösteriyor
(çünkü hepsi trend taşıyor) ama gerçek ay-ay hareketlere inince noter
devriyle bağları güçlü değil.

**4) Tüketici güveni endeksinin noter devriyle neredeyse HİÇ ilişkisi
yok** (r ≈ -0,03, hem ham seviye hem log-değişimde) — "insanlar genel
ekonomiye güveniyor mu" sorusunun cevabı ile "kaç araç el değiştirdi"
arasında beklenen bağ bu veride görünmüyor; bu, ekip lideriyle tartışılmaya
değer, sezgiye aykırı bir bulgu.

**5) Örneklem büyüklüğü uyarısı.** Log-değişim korelasyonları ~100
gözlemle hesaplandı — bu projenin diğer analizlerine göre orta-iyi bir
örneklem, ama yine de bu notebook'ta p-değeri veya çoklu-test düzeltmesi
HESAPLANMADI. Buradaki r değerleri "kesin kanıt" değil, "izlenmeye değer
sinyal" olarak okunmalı.

**Olası sonraki adımlar (yalnızca öneri, karar değil):**
- `osd_binek_adet` ve `odmd_otomobil_adet`'in noter devriyle olan
  ilişkisi, GECİKMELİ (bu ayki üretim, gelecek ayki devri mi etkiliyor?)
  olarak da test edilebilir.
- Bu analiz, DF-A'nın "tam kapsamlı" (2015'e kadar giden) versiyonuyla da
  tekrarlanıp örneklem büyütülebilir.
- DF-B'deki fiyat/ENAG sinyalleriyle birleştirilip daha zengin bir tablo
  üzerinde de benzer bir analiz yapılabilir.
